<!--
SPDX-FileCopyrightText: Copyright (c) 2026, NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: Apache-2.0
-->

# Varying the harness, capabilities, and telemetry

The [quickstart](01_quickstart.ipynb) ran one agent on one harness. That by
itself is not a reason to adopt Fabric. The reason is this: once an agent is
a typed config, you can change **how** it runs without touching your
application code. This notebook takes one code-review agent and varies it
along the three axes Fabric is built for:

1. **Harness** -- run the same agent on Hermes, Deep Agents, Codex, or Claude.
2. **Capabilities** -- add or remove skills and MCP servers.
3. **Telemetry** -- attach NeMo Relay tracing and get trace files out.

Where a harness's prerequisites are present in this environment, the cells
below **actually run it**. Where they are not, the cell inspects the
resolved config with `plan()` and tells you exactly what to provide to run
it for real -- so the notebook always executes top to bottom.

<div align="center">
<img src="img/three-axes.svg" width="620" alt="Switch and configure one agent along three axes.">
<br><br>
<em>One typed agent, varied three ways — harness, capabilities, and telemetry — without changing your application code.</em>
</div>

## Setup

Load the environment and the code-review agent's config builders. A harness
adapter runs in whatever Python environment has that adapter installed;
`settings["python"]` selects it. Here the Hermes adapter lives in the repo's
Hermes venv, and the others live in this notebook's own interpreter.

In [ ]:
import importlib.util
import os
import shutil
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    root = start.resolve()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent
    return root


def load_dotenv(path: Path) -> None:
    if not path.exists():
        return
    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.removeprefix("export ").partition("=")
        key = key.strip()
        if key and key not in os.environ:
            os.environ[key] = value.strip().strip("'\"")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
load_dotenv(REPO_ROOT / ".env")

# Each harness adapter runs in some Python environment. The Hermes adapter lives
# in the repo's separate Hermes venv; the Codex, Claude, and Deep Agents adapters
# live alongside Fabric in this notebook's own interpreter.
FABRIC_PY = sys.executable
_hermes_python = REPO_ROOT / ".tmp" / "hermes-venv" / "bin" / "python"
HERMES_PY = str(_hermes_python) if _hermes_python.exists() else os.environ.get("ADAPTER_PYTHON")

from nemo_fabric import Fabric
from examples.code_review_agent import (
    BASE_DIR,
    claude_config,
    codex_cli_config,
    deepagents_config,
    hermes_sdk_config,
    with_relay,
)

fabric = Fabric()
RELAY_AVAILABLE = importlib.util.find_spec("nemo_relay") is not None
print("fabric interpreter :", FABRIC_PY)
print("hermes interpreter :", HERMES_PY or "not found")
print("nemo_relay present :", RELAY_AVAILABLE)


## Axis 1 -- Run the same agent on different harnesses

One agent, one prompt, four harnesses. Each builder differs only in the
adapter it selects and that harness's settings; Fabric resolves each through
the same contract and returns the same result shape. Because the harnesses
wrap different models, the **replies differ** -- that is the point.

The helpers below pin each adapter's interpreter, decide whether the harness
can run here, and either run it or explain what is missing.

In [ ]:
# Each harness needs a Python env with its adapter installed, plus credentials.
# `settings["python"]` picks the interpreter that runs the adapter.
HARNESSES = [
    {"name": "hermes-sdk", "builder": hermes_sdk_config, "python": HERMES_PY,
     "key": "NVIDIA_API_KEY",
     "needs": "a Hermes install (repo README's Hermes SDK quick start) + NVIDIA_API_KEY"},
    {"name": "deepagents", "builder": deepagents_config, "python": FABRIC_PY,
     "key": "NVIDIA_API_KEY",
     "needs": "the Deep Agents adapter (nemo-fabric[deepagents]) + NVIDIA_API_KEY"},
    {"name": "codex-cli", "builder": codex_cli_config, "python": FABRIC_PY,
     "key": "OPENAI_API_KEY", "binary": "codex",
     "needs": "an authenticated Codex CLI (`codex`) + OPENAI_API_KEY"},
    {"name": "claude", "builder": claude_config, "python": FABRIC_PY,
     "key": "ANTHROPIC_API_KEY",
     "needs": "the Claude adapter + ANTHROPIC_API_KEY"},
]


def prepared(entry):
    """Build the harness config with its adapter interpreter pinned."""
    cfg = entry["builder"]()
    if entry.get("python"):
        cfg.harness.settings["python"] = entry["python"]
    return cfg


def blocker(entry):
    """Return why a harness cannot run here, or None if it can."""
    py = entry.get("python")
    if not py or not os.path.exists(py):
        return "adapter interpreter not found"
    if entry.get("key") and not os.environ.get(entry["key"]):
        return f"{entry['key']} is not set"
    if entry.get("binary") and not shutil.which(entry["binary"]):
        return f"`{entry['binary']}` not on PATH"
    return None


def oneline(text, limit=220):
    s = " ".join(str(text).split())
    return s if len(s) <= limit else s[:limit] + " ..."


In [ ]:
PROMPT = "In one sentence, what does the Python expression sum(v) / len(v) compute, and name one risk?"

for entry in HARNESSES:
    cfg = prepared(entry)
    plan = fabric.plan(cfg, base_dir=BASE_DIR)
    model = plan.effective_config.config.models["default"]["model"]
    rc = plan.effective_config.config.runtime
    print(f"### {entry['name']}")
    print(f"    adapter = {plan.adapter.adapter_id}")
    print(f"    model   = {model}   io = {rc.input_schema}/{rc.output_schema}")
    reason = blocker(entry)
    if reason:
        print(f"    NOT RUN here ({reason}).")
        print(f"    To run it, provide: {entry['needs']}")
    else:
        try:
            result = await fabric.run(cfg, base_dir=BASE_DIR, input=PROMPT)
            reply = getattr(result.output, "response", result.output)
            print(f"    RAN -> {result.status}")
            print(f"    reply: {oneline(reply)}")
        except Exception as error:
            print(f"    run failed ({type(error).__name__}: {oneline(error, 120)}).")
            print(f"    To run it, provide: {entry['needs']}")
    print()

## Axis 2 -- Add and remove capabilities

Skills and MCP servers are normalized capabilities. You compose them onto a
**copy** of a config with `add_skill_path` / `remove_skill_path` and
`add_mcp_server` / `remove_mcp_server`, and `resolve()` shows the effective
config the harness would receive. That resolved config *is* the difference:
it is exactly the capability set handed to the harness at launch.

Editing a copy never touches the base, so you can build many capability
variants from one agent.

In [ ]:
def capabilities(cfg):
    """Return the skills and MCP servers in a config's resolved form."""
    resolved = fabric.resolve(cfg, base_dir=BASE_DIR).config
    skills = (resolved.get("skills") or {}).get("paths", [])
    servers = list((resolved.get("mcp") or {}).get("servers", {}))
    return {"skills": skills, "mcp_servers": servers}

# The Hermes SDK agent ships with one skill and one MCP server.
base = hermes_sdk_config()
print("base         :", capabilities(base))

# Strip both on a copy.
stripped = hermes_sdk_config()
stripped.remove_skill_path("./skills/code-review")
stripped.remove_mcp_server("github")
print("stripped     :", capabilities(stripped))

# Add a second skill and a second MCP server on another copy.
extended = hermes_sdk_config()
extended.add_skill_path("./skills/style-guide")
extended.add_mcp_server(
    "docs", transport="streamable-http", url="${DOCS_MCP_URL}", exposure="fabric_managed"
)
print("+skill +mcp  :", capabilities(extended))

# The base was never mutated.
print("base intact  :", capabilities(base))

## Axis 3 -- Turn on telemetry (NeMo Relay)

`with_relay(...)` returns a copy of a config with NeMo Relay tracing enabled.
Telemetry rides the same typed config and the same run call -- nothing else
about the agent changes. When the Relay dependency is present in the adapter
environment, the run **writes trace files** you can collect afterward.

We run the Deep Agents variant (its adapter environment here has `nemo_relay`)
and then print the full trace it produced -- every ATOF event.

In [ ]:
import json

relay_dir = BASE_DIR / "artifacts" / "relay"
deep = next(e for e in HARNESSES if e["name"] == "deepagents")
reason = blocker(deep)

if reason or not RELAY_AVAILABLE:
    missing = reason or "nemo_relay is not installed in the adapter environment"
    print(f"Not emitting telemetry here ({missing}).")
    print("Install the Relay extra (nemo-fabric[relay]) in the adapter env to")
    print("produce ATOF/ATIF trace files under", relay_dir)
else:
    if relay_dir.exists():
        shutil.rmtree(relay_dir)
    cfg = with_relay(deepagents_config())
    cfg.harness.settings["python"] = deep["python"]
    try:
        result = await fabric.run(cfg, base_dir=BASE_DIR, input=PROMPT)
    except Exception as error:
        print(f"Telemetry run failed ({type(error).__name__}: {oneline(error, 120)}).")
        print("To emit traces, provide:", deep["needs"])
    else:
        print("run status     :", result.status)
        print("telemetry refs :", [t.provider for t in result.telemetry] or "(none)")

        trace_files = sorted(p for p in relay_dir.rglob("*") if p.is_file())
        print("trace files    :", len(trace_files))
        for path in trace_files:
            print("   ", path.relative_to(REPO_ROOT))

        # Print the full Relay trace: every ATOF event, plus any ATIF trajectory.
        for path in trace_files:
            print(f"\n===== {path.relative_to(REPO_ROOT)} =====")
            text = path.read_text()
            if path.name.endswith(".jsonl"):
                lines = [line for line in text.splitlines() if line.strip()]
                for i, line in enumerate(lines, 1):
                    print(f"\n--- event {i}/{len(lines)} ---")
                    print(json.dumps(json.loads(line), indent=2))
            else:
                print(json.dumps(json.loads(text), indent=2))

## Recap

These three axes are the reason to reach for Fabric:

- **One agent, many harnesses.** The same `FabricConfig` ran on different
  runtimes and returned the same result shape, so your application code does
  not change when you switch or compare runtimes.
- **Capabilities as config.** Skills and MCP servers are added or removed on
  a copy of the config, so you compose exactly the capability set each run
  needs -- the basis for ablation studies.
- **Uniform telemetry.** Enabling Relay produced trace files through the same
  contract, no matter which harness ran.

Together they are what make Fabric useful for real work: comparing runtimes,
running evaluation and ablation sweeps, and observing agents in production --
all without rewriting the agent. For the full API, see the
[Python SDK guide](../../docs/sdk/python.mdx).